In [ ]:
import pandas as pd
import pycountry
import wbdata

In [125]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
            "prepare_wacc",
    )

### File paths

In [126]:
wacc_fn = snakemake.input.wacc
bus_locations_fn = snakemake.input.bus_locations

# Output
wacc_clustered_fn = snakemake.output.wacc

# config
config = snakemake.config

### Get WACC data

In [127]:
# Read the CSV data
wacc = pd.read_csv(wacc_fn, comment='#', index_col="iso_alpha")


### Cluster per region


Map each country in the production data to its region using the region definitions from the config file.


In [128]:
# Load region definitions from config file
regions = config['regions']

In [129]:
country_name_corrections = {
    "Democratic Republic of the Congo": "Congo, The Democratic Republic of the",
    "Republic of the Congo": "Republic of the Congo",
    "Kosovo": "Republic of Kosovo",  # pycountry not supported
    "Russia": "Russian Federation",
    "Turkey": "Türkiye",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Tanzania": "United Republic of Tanzania",
    "Bolivia": "Plurinational State of Bolivia",
    "Vietnam": "Viet Nam",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Taiwan": "Taiwan, Province of China",
    "Laos": "Lao People's Democratic Republic",
    "Brunei": "Brunei Darussalam",
    "Equatorial French Guiana": "French Guiana",
    "Syria": "Syrian Arab Republic",   
    "Palestine": "Palestine, State of",
    "Moldova": "Republic of Moldova",
    "North Korea": "Korea, Democratic People's Republic of",
}

In [130]:
# Apply a country name correction to the regions dict
for region, countries in regions.items():
    for i, country in enumerate(countries):
        if country in country_name_corrections:
            countries[i] = country_name_corrections[country]

In [131]:
# Build a mapping from country name to ISO_A2 code
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_3
    # Add common names
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_3

# Build a mapping from ISO_A2 code to region
iso_to_region = {}
for region, countries in regions.items():
    for country in countries:
        # Some country names may have extra text (e.g., "Togo + Algeria"), handle them simply
        for part in country.split('+'):
            name = part.strip()
            code = country_name_to_iso.get(name)
            if code:
                iso_to_region[code] = region

In [132]:
# Map each row in production to its region
wacc['region'] = wacc.index.map(lambda iso: iso_to_region.get(iso, 'Other'))

### 

### Cluster: Weighted by GDP

In [ ]:
# --- fetch GDP (current USD) from World Bank ---
gdp_raw = wbdata.get_dataframe(
    {"NY.GDP.MKTP.CD": "gdp_usd"},
    date="2024",
)
gdp = gdp_raw.dropna(subset=["gdp_usd"]).reset_index()
gdp = gdp.rename(columns={"country": "country_name"})

# map country name → ISO-3 so we can merge on the wacc index
gdp["country_code"] = gdp["country_name"].map(
    lambda name: country_name_to_iso.get(name)
)
gdp = gdp.dropna(subset=["country_code"])

# --- attach GDP weights ---
wacc_gdp = wacc.merge(gdp[["country_code", "gdp_usd"]], left_index=True, right_on="country_code", how="left")

# fill missing GDP with regional median so no country is dropped
wacc_gdp["gdp_usd"] = wacc_gdp.groupby("region")["gdp_usd"].transform(
    lambda x: x.fillna(x.median())
)

def weighted_mean(df, value_col, weight_col):
    return (df[value_col] * df[weight_col]).sum() / df[weight_col].sum()

wacc_by_region = (
    wacc_gdp.groupby("region")
    .apply(lambda g: pd.Series({
        "wacc":                  weighted_mean(g, "wacc", "gdp_usd"),
        "wacc_real":             weighted_mean(g, "wacc_real", "gdp_usd"),
        "risk_free_rate":        weighted_mean(g, "risk_free_rate", "gdp_usd"),
        "country_risk_premium":  weighted_mean(g, "country_risk_premium", "gdp_usd"),
        "equity_risk_premium":   weighted_mean(g, "equity_risk_premium", "gdp_usd"),
        "cost_of_equity":        weighted_mean(g, "cost_of_equity", "gdp_usd"),
        "cost_of_debt":          weighted_mean(g, "cost_of_debt", "gdp_usd"),
        "n_countries":           len(g),
        "wacc_spread":           g["wacc"].max() - g["wacc"].min(),  # diagnostics
    }))
    .reset_index()
)

/tmp/ipykernel_30395/4062138612.py:30: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [134]:
wacc_by_region.set_index("region", inplace=True)

### Save clustered WACC

In [135]:
wacc_by_region

,wacc,wacc_real,risk_free_rate,country_risk_premium,equity_risk_premium,cost_of_equity,cost_of_debt,n_countries,wacc_spread
region,,,,,,,,,
Central_America,0.109872,0.088110,0.035,0.034729,0.065,0.221420,0.035508,11.0,0.076141
East_Asia,0.104929,0.083263,0.035,0.008741,0.065,0.203540,0.039187,4.0,0.025600
East_East_Asia,0.100051,0.078482,0.035,0.008000,0.065,0.195985,0.036095,2.0,0.003043
Eurasia,0.118485,0.096554,0.035,0.038948,0.065,0.235939,0.040182,8.0,0.033607
Europe,0.105320,0.083647,0.035,0.013418,0.065,0.206017,0.038189,33.0,0.103248
Far_West_Europe,0.103869,0.082225,0.035,0.010734,0.065,0.202772,0.037934,7.0,0.009259
Middle_East,0.128845,0.106711,0.035,0.063994,0.065,0.261493,0.040413,14.0,0.099423
North_America,0.098852,0.077306,0.035,0.000000,0.065,0.190962,0.037446,2.0,0.001111
North_West_Africa,0.131542,0.109355,0.035,0.079286,0.065,0.271699,0.038105,20.0,0.083570


In [136]:
# Filter for relevant columns
wacc_by_region = wacc_by_region[["wacc_real", "n_countries", "wacc_spread"]]

In [137]:
wacc_by_region.to_csv(wacc_clustered_fn)

### Mean (depreciated)

In [138]:
# wacc_by_region = wacc.groupby('region').mean(numeric_only=True)
# wacc_by_region.round(3)